# CipherMark : étape 1, les mesures restantes du rapport d'examen

**Exécution → Modifier le type d'exécution → T4 GPU**, puis **Exécution → Tout exécuter**.

| Section | Point du rapport | Durée estimée (T4) |
|---|---|---|
| A. Seuil et marge du hash, calibrage conjoint | C3, transplantation | 5 min |
| B. Latence de la chaîne post-hoc | Q5.1 | 5 min |
| C. Autorégressif sur images jamais vues, verdict complet | C4 | 40 min |
| D. Recadrage 0,9 + resynchronisation, verdict complet | C2 | 30 min |
| E. Diffusion, verdict complet (512 px, COCO val2017) | tableau de clôture | 1 h 30 à 2 h |

Chaque section copie ses résultats sur le Drive (`ciphermark/14-colab-etape1/resultats/`) dès qu'elle finit,
et **saute** si son résultat y est déjà : si la session est coupée, rouvrir et refaire « Tout exécuter ».

In [ ]:
import torch, subprocess
print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout)
if not torch.cuda.is_available():
    raise SystemExit("Pas de GPU : Execution -> Modifier le type d'execution -> T4, puis relancer.")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys, shutil, subprocess

RACINE_DRIVE = "/content/drive/MyDrive/ciphermark"
DOSSIER      = f"{RACINE_DRIVE}/14-colab-etape1"
SORTIE       = f"{DOSSIER}/resultats"
DEPOT        = "/content/code-memoire"
os.makedirs(SORTIE, exist_ok=True)

if not os.path.isdir(os.path.join(DEPOT, "distseal", "utils")):
    tmp = "/content/_extraction"
    shutil.rmtree(tmp, ignore_errors=True); os.makedirs(tmp)
    subprocess.run(["tar", "xzf", f"{RACINE_DRIVE}/00-code-source-a-jour.tar.gz", "-C", tmp], check=True)
    racine = next(r for r, _, _ in os.walk(tmp) if os.path.isdir(os.path.join(r, "distseal", "utils")))
    shutil.rmtree(DEPOT, ignore_errors=True); shutil.move(racine, DEPOT)

# scripts plus recents que l'archive
for s in ("diag_marge_hash.py", "diag_resync_authentic.py", "mesure_latence.py",
          "eval_grande_echelle_decodeur.py"):
    shutil.copy(f"{DOSSIER}/{s}", f"{DEPOT}/scripts/ciphermark/{s}")

# tatoueur de reference 64 bits (MD5 b4dbbfdf2302...) et conditionneur
# autoregressif de la campagne (phase F, pas 60 000)
os.makedirs(f"{DEPOT}/runs", exist_ok=True); os.makedirs("/content/ckpt", exist_ok=True)
CKPT    = f"{DEPOT}/runs/ciphermark_64bits_checkpoint.pth"
COND_AR = "/content/ckpt/phaseF_maskgit.pt"
for dst, src in ((CKPT, "01-tatoueur-modeles-de-reference-64-et-128-bits/phaseA2_64bits_stable_checkpoint.pth"),
                 (COND_AR, "06-entrainements-en-cours-non-termines/phaseF_maskgit/checkpoint.pt")):
    if not os.path.exists(dst):
        shutil.copy(f"{RACINE_DRIVE}/{src}", dst)

# corpus d'evaluation : COCO val2017 a 256 px, jamais vu a l'entrainement
if not os.path.isdir("/content/corpus-coco-5000/toutes"):
    subprocess.run(["tar", "xzf", f"{RACINE_DRIVE}/13-colab-diagnostics-c2-c3/corpus-coco-5000.tar.gz",
                    "-C", "/content"], check=True)
os.chdir(DEPOT)
print("images :", len(os.listdir("/content/corpus-coco-5000/toutes")), "| pret")

In [ ]:
import importlib.util
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lpips", "torchmetrics",
                "einops", "omegaconf", "safetensors", "torchdiffeq"])
for mod, paquet in (("reedsolo", "reedsolo"), ("Crypto", "pycryptodome"),
                    ("huggingface_hub", "huggingface_hub")):
    if importlib.util.find_spec(mod) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", paquet])

ENV = dict(os.environ, PYTHONPATH=f"{DEPOT}/deps:{DEPOT}")

def lance(nom, cmd, produits):
    """Lance une mesure, garde son journal, copie ses produits sur le Drive.
    Saute si le premier produit est deja sur le Drive."""
    cible = os.path.join(SORTIE, os.path.basename(produits[0]))
    fait = (any(f.endswith(".json") for f in os.listdir(cible)) if os.path.isdir(cible)
            else os.path.exists(cible))
    if fait:
        print(f"{nom} : deja sur le Drive, saute"); return
    journal = f"{DEPOT}/runs/{nom}.log"
    with open(journal, "w") as f:
        p = subprocess.Popen(cmd, env=ENV, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for ligne in p.stdout:
            f.write(ligne)
            if "warn" not in ligne.lower():
                print(ligne, end="")
    code = p.wait()
    print(f"\ncode de sortie : {code}" + ("  (-9 : tue, souvent par manque de RAM)" if code == -9 else ""))
    for fic in [journal] + produits:
        if os.path.isdir(fic):
            shutil.copytree(fic, os.path.join(SORTIE, os.path.basename(fic)), dirs_exist_ok=True)
        elif os.path.exists(fic):
            shutil.copy(fic, SORTIE)
    print("copie sur le Drive :", SORTIE)
print("dependances installees")

## A. C3 : seuil et marge du hash, calibrage conjoint (5 min)

In [ ]:
lance("A_marge_hash",
      [sys.executable, "scripts/ciphermark/diag_marge_hash.py", "--n-images", "1000",
       "--batch", "32", "--threads", "2", "--corpus", "/content/corpus-coco-5000"],
      [f"{DEPOT}/runs/diag_marge_hash_N1000.json", f"{DEPOT}/runs/diag_marge_hash_N1000_paires.png",
       f"{DEPOT}/runs/diag_marge_hash_N1000_bruts.npz"])

## B. Q5.1 : latence de la chaîne post-hoc (5 min)

In [ ]:
lance("B_latence",
      [sys.executable, "scripts/ciphermark/mesure_latence.py", "--n", "50",
       "--corpus", "/content/corpus-coco-5000"],
      [f"{DEPOT}/runs/mesure_latence.json"])

## C. C4 : autorégressif sur images jamais vues, 5 000 images, 20 conditions (40 min)

In [ ]:
lance("C_autoregressif_coco5000",
      [sys.executable, "scripts/ciphermark/eval_grande_echelle_decodeur.py",
       "--famille", "autoregressif", "--source", "encode",
       "--conditionneur", COND_AR, "--watermarker", CKPT,
       "--corpus", "/content/corpus-coco-5000", "--n-images", "5000", "--batch", "16",
       "--user-id", "createur-0042", "--out-dir", f"{DEPOT}/runs/C_autoregressif_coco5000"],
      [f"{DEPOT}/runs/C_autoregressif_coco5000"])

## D. C2 : recadrage 0,9 + resynchronisation, verdict complet, 500 images (30 min)

In [ ]:
lance("D_resync_authentic",
      [sys.executable, "scripts/ciphermark/diag_resync_authentic.py", "--n-images", "500",
       "--corpus", "/content/corpus-coco-5000"],
      [f"{DEPOT}/runs/diag_resync_authentic_N500_c0.9.json"])

## E. Diffusion, verdict complet, 5 000 images à 512 px (1 h 30 à 2 h)

La campagne du 31 août ne mesurait que la détection statistique pour le décodeur conditionné.
Cette section mesure le verdict AUTHENTIC complet, sur COCO val2017 à 512 px (téléchargé ici, environ 800 Mo).
Conditionneur : phaseD512_degele, pas 20 000, celui de la campagne (6 Go, copié depuis le Drive).

In [ ]:
COND_DIFF = "/content/ckpt/phaseD512_degele.pt"
if not os.path.exists(COND_DIFF):
    shutil.copy(f"{RACINE_DRIVE}/06-entrainements-en-cours-non-termines/"
                "phaseD512-degele-decodeur-a-512px/checkpoint.pt", COND_DIFF)
CORPUS512 = "/content/corpus-coco-5000-512"
n512 = sum(len(f) for _, _, f in os.walk(CORPUS512)) if os.path.isdir(CORPUS512) else 0
if n512 < 5000:
    subprocess.run([sys.executable, "scripts/ciphermark/build_corpus.py", "--out", CORPUS512,
                    "--size", "512", "--coco", "val2017", "--coco-max", "5000",
                    "--tmp-dir", "/content"], env=ENV, check=True)
lance("E_diffusion_coco5000_512",
      [sys.executable, "scripts/ciphermark/eval_grande_echelle_decodeur.py",
       "--famille", "diffusion", "--source", "encode",
       "--conditionneur", COND_DIFF, "--watermarker", CKPT,
       "--corpus", CORPUS512, "--n-images", "5000", "--batch", "4",
       "--user-id", "createur-0042", "--out-dir", f"{DEPOT}/runs/E_diffusion_coco5000_512"],
      [f"{DEPOT}/runs/E_diffusion_coco5000_512"])